In [1]:
# ==========================================
# AI-POWERED FX RISK INTELLIGENCE PLATFORM
# WEEK 4 - SESSION 3
# Model Overfitting, Weaknesses and Final Model Selection
# ==========================================

# Import libraries
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

from sklearn.metrics import (mean_squared_error,mean_absolute_error,r2_score)

# Load previous results
results = pd.read_csv("../outputs/evaluation/walk_forward_results.csv")
performance = pd.read_csv("../outputs/evaluation/walk_forward_performance.csv")

# Check results
print("Prediction results:")
print(results.shape)
print(results.head())

print("Model performance:")
print(performance.shape)
print(performance.head())

print("Currency pairs:")
print(results["Currency"].unique())

print("Models:")
print(results["Model"].unique())

Prediction results:
(2214, 6)
  Currency          Model  Fold        Date    Actual  Predicted
0   GBPUSD  Random Forest     1  2025-05-23  0.006116   0.005552
1   GBPUSD  Random Forest     1  2025-05-26  0.004521   0.005703
2   GBPUSD  Random Forest     1  2025-05-27  0.005616   0.005533
3   GBPUSD  Random Forest     1  2025-05-28  0.005355   0.005283
4   GBPUSD  Random Forest     1  2025-05-29  0.004019   0.005051
Model performance:
(18, 7)
  Currency          Model  Fold       MAE      RMSE        R2  \
0   GBPUSD  Random Forest     1  0.001360  0.001658 -0.498617   
1   GBPUSD  Random Forest     2  0.001266  0.001612 -0.092534   
2   GBPUSD  Random Forest     3  0.001449  0.001661 -0.611165   
3   GBPUSD        XGBoost     1  0.001466  0.001807 -0.781455   
4   GBPUSD        XGBoost     2  0.001237  0.001593 -0.066960   

   Unique_Predictions  
0                 123  
1                 123  
2                 123  
3                 123  
4                 123  
Currency pairs:
<S

In [6]:
# Reload previous session modelling environment

# Import libraries
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Load original feature dataset
df = pd.read_csv("../data/processed/ml_features.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Define target variables
target_columns = [
    "GBPUSD_Target_Volatility_5D",
    "EURUSD_Target_Volatility_5D",
    "USDJPY_Target_Volatility_5D"
]

# Identify predictive features
feature_columns = [
    col for col in df.columns
    if col not in ["Date"] + target_columns
]

feature_columns = (
    df[feature_columns]
    .select_dtypes(include=[np.number])
    .columns.tolist()
)

# Create modelling dataset
df_model = df.dropna(
    subset=feature_columns + target_columns
).copy()

# Define independent variables
X = df_model[feature_columns]

# Define dependent variables
y = {
    "GBPUSD": df_model["GBPUSD_Target_Volatility_5D"],
    "EURUSD": df_model["EURUSD_Target_Volatility_5D"],
    "USDJPY": df_model["USDJPY_Target_Volatility_5D"]
}

# Extract dates
dates = df_model["Date"]

# Recreate validation configuration
n = len(X)

initial_train_size = int(n * 0.70)
test_size = int(n * 0.10)
gap = 5

# Recreate expanding-window validation folds
validation_periods = []

for fold in range(3):

    test_start = initial_train_size + (fold * test_size)
    test_end = test_start + test_size

    train_end = test_start - gap

    train_indices = np.arange(0, train_end)
    test_indices = np.arange(test_start, test_end)

    validation_periods.append({
        "fold": fold + 1,
        "train_indices": train_indices,
        "test_indices": test_indices
    })

# Recreate Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1
)

# Recreate XGBoost model
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    object="reg:squarederror",
    random_state=42,
    n_jobs=1
)

# Store model configurations
models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

print("Session modelling environment restored")
print("-----------------------------------------")
print("Feature dataset:", X.shape)
print("Currency targets:", list(y.keys()))
print("Validation folds:", len(validation_periods))
print("Models:", list(models.keys()))
print("Training gap:", gap)

Session modelling environment restored
-----------------------------------------
Feature dataset: (1235, 21)
Currency targets: ['GBPUSD', 'EURUSD', 'USDJPY']
Validation folds: 3
Models: ['Random Forest', 'XGBoost']
Training gap: 5


In [7]:
# Validate results

# 1. Check for missing values
print("Missing values:")
print(results.isna().sum())

# 2. Check the number of predictions per currency and model
print("Predictions by currency and model:")
print(results.groupby(["Currency", "Model"]).size())

# 3. Check predictions per validation fold
print("Prediction by fold:")
print(results.groupby(["Currency", "Model", "Fold"]).size())

# 4. Recalculate RMSE and MAE from predictions
recalculated = (results.groupby(["Currency", "Model", "Fold"]).apply(lambda group: pd.Series({"Recalculated_MAE": mean_absolute_error(group["Actual"], group["Predicted"]), "Recalculated_RMSE": np.sqrt(mean_squared_error(group["Actual"], group["Predicted"]))}), include_groups=False).reset_index())
print("Recalculated performance:")
print(recalculated.round(6))

# 5. Compare against the saved performance metrics
comparison = performance.merge(recalculated,on=["Currency", "Model", "Fold"])
comparison["RMSE_Difference"] = (comparison["RMSE"] - comparison["Recalculated_RMSE"])
comparison["MAE_Difference"] = (comparison["MAE"] - comparison["Recalculated_MAE"])
print("Metric validation:")
print(comparison[["Currency", "Model", "Fold", "RMSE_Difference", "MAE_Difference"]].round(10))

Missing values:
Currency     0
Model        0
Fold         0
Date         0
Actual       0
Predicted    0
dtype: int64
Predictions by currency and model:
Currency  Model        
EURUSD    Random Forest    369
          XGBoost          369
GBPUSD    Random Forest    369
          XGBoost          369
USDJPY    Random Forest    369
          XGBoost          369
dtype: int64
Prediction by fold:
Currency  Model          Fold
EURUSD    Random Forest  1       123
                         2       123
                         3       123
          XGBoost        1       123
                         2       123
                         3       123
GBPUSD    Random Forest  1       123
                         2       123
                         3       123
          XGBoost        1       123
                         2       123
                         3       123
USDJPY    Random Forest  1       123
                         2       123
                         3       123
          XGBoost 

In [8]:
# Check available variables
print("Available variables:")
print("X:", X.shape)
print("Targets:", list(y.keys()))
print("Validation folds:", len(validation_periods))
print("Models:", list(models.keys()))

Available variables:
X: (1235, 21)
Targets: ['GBPUSD', 'EURUSD', 'USDJPY']
Validation folds: 3
Models: ['Random Forest', 'XGBoost']


In [9]:
# Training error analysis 
from sklearn.base import clone 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

training_results = []
for currency, target in y.items():
    print(f"\nProcessing {currency}")
    for model_name, model in models.items():
        for period in validation_periods:
            fold = period["fold"]
            train_idx = period["train_indices"]
            X_train = X.iloc[train_idx]
            y_train = target.iloc[train_idx]

            # Fit a fresh model using the training data
            fitted_model = clone(model)
            fitted_model.fit(X_train, y_train)

            # Predict on training data
            train_predictions = fitted_model.predict(X_train)

            # Calculate training performance
            train_mae = mean_absolute_error(y_train, train_predictions)
            train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
            train_r2 = r2_score(y_train, train_predictions)
            training_results.append({
                "Currency": currency,
                "Model": model_name,
                "Fold": fold,
                "Training_Size": len(train_idx),
                "Train_MAE": train_mae,
                "Train_RMSE": train_rmse,
                "Train_R2": train_r2
            })

# Convert results into DataFrame
training_performance = pd.DataFrame(training_results)
print("\nTraining performance results:")
print(training_performance.round(6).to_string(index=False))


Processing GBPUSD


c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Processing EURUSD


c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Processing USDJPY


c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Training performance results:
Currency         Model  Fold  Training_Size  Train_MAE  Train_RMSE  Train_R2
  GBPUSD Random Forest     1            859   0.001178    0.001543  0.717934
  GBPUSD Random Forest     2            982   0.001195    0.001546  0.693003
  GBPUSD Random Forest     3           1105   0.001194    0.001545  0.670052
  GBPUSD       XGBoost     1            859   0.001148    0.001444  0.752864
  GBPUSD       XGBoost     2            982   0.001147    0.001442  0.732863
  GBPUSD       XGBoost     3           1105   0.001155    0.001449  0.709931
  EURUSD Random Forest     1            859   0.001042    0.001333  0.642719
  EURUSD Random Forest     2            982   0.001044    0.001340  0.629558
  EURUSD Random Forest     3           1105   0.001048    0.001359  0.611802
  EURUSD       XGBoost     1            859   0.000999    0.001268  0.676596
  EURUSD       XGBoost     2            982   0.001028    0.001317  0.642349
  EURUSD       XGBoost     3           1105  

c:\Users\alexb\fx-risk-intelligence\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:41:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [10]:
# Merge training and testing performance
generalisation = training_performance.merge(performance, on=["Currency", "Model", "Fold"])

# Calculate generalisation gaps
generalisation["RMSE_Gap"] = (generalisation["RMSE"] - generalisation["Train_RMSE"])
generalisation["MAE_Gap"] = (generalisation["MAE"] - generalisation["Train_MAE"])

# Calculate test-to-training RMSE ratio 
generalisation["RMSE_Ratio"] = (generalisation["RMSE"] / generalisation["Train_RMSE"])

# Display results
print("Generalisation gap results:")
print(generalisation[["Currency", "Model", "Fold", "Train_RMSE", "RMSE", "RMSE_Gap", "RMSE_Ratio"]].round(6).to_string(index=False))

# Average generalisation gap by currency and model
gap_summary = (generalisation.groupby(["Currency", "Model"])[["Train_RMSE", "RMSE", "RMSE_Gap", "RMSE_Ratio"]].mean().reset_index())
print("\nAverage generalisation gap:")
print(gap_summary.round(6).to_string(index=False))

Generalisation gap results:
Currency         Model  Fold  Train_RMSE     RMSE  RMSE_Gap  RMSE_Ratio
  GBPUSD Random Forest     1    0.001543 0.001658  0.000115    1.074225
  GBPUSD Random Forest     2    0.001546 0.001612  0.000066    1.042698
  GBPUSD Random Forest     3    0.001545 0.001661  0.000115    1.074670
  GBPUSD       XGBoost     1    0.001444 0.001807  0.000363    1.251251
  GBPUSD       XGBoost     2    0.001442 0.001593  0.000151    1.104625
  GBPUSD       XGBoost     3    0.001449 0.001591  0.000142    1.098119
  EURUSD Random Forest     1    0.001333 0.002108  0.000775    1.581764
  EURUSD Random Forest     2    0.001340 0.001904  0.000563    1.420325
  EURUSD Random Forest     3    0.001359 0.001715  0.000356    1.262213
  EURUSD       XGBoost     1    0.001268 0.002255  0.000987    1.778811
  EURUSD       XGBoost     2    0.001317 0.001966  0.000649    1.493018
  EURUSD       XGBoost     3    0.001352 0.001815  0.000463    1.342242
  USDJPY Random Forest     1    0.00

In [14]:
# Calculate prediction errors
results["Forecast_Error"] = results["Predicted"] - results["Actual"]

results["Absolute_Error"] = results["Forecast_Error"].abs()

# Calculate percentage of overpredictions
results["Overpredictions"] = results["Forecast_Error"] > 0

# Group results by currency and model 
bias_analysis = (
    results
    .groupby(["Currency", "Model"])
    .agg(
        Mean_Bias=("Forecast_Error", "mean"),
        Mean_Absolute_Error=("Absolute_Error", "mean"),
        Error_Std=("Forecast_Error", "std"),
        Overprediction_Rate=("Overpredictions", "mean"),
        Observations=("Forecast_Error", "count")
    )
    .reset_index()
)

# Convert overprediction rate to percentage
bias_analysis["Overprediction_Rate"] *= 100

print("Forecast bias analysis:")
print(bias_analysis.round(6).to_string(index=False))
results["Absolute_Error"] = (results["Forecast_Error"].abs())

# Calculate percentage of overpredictions
results["Overpredictions"] = (results["Forecast_Error"] > 0)

# Group results by currency and model 
bias_analysis = (
    results
    .groupby(["Currency", "Model"])
    .agg(
        Mean_Bias=("Forecast_Error", "mean"),
        Mean_Absolute_Error=("Absolute_Error", "mean"),
        Error_Std=("Forecast_Error", "std"),
        Overprediction_Rate=("Overpredictions", "mean"),
        Observations=("Forecast_Error", "count")
    )
    .reset_index()
)

# Convert overprediciton rate to percentage
bias_analysis["Overprediction_Rate"] *= 100
print("Forecast bias analysis:")
print(bias_analysis.round(6).to_string(index=False))

Forecast bias analysis:
Currency         Model  Mean_Bias  Mean_Absolute_Error  Error_Std  Overprediction_Rate  Observations
  EURUSD Random Forest   0.000831             0.001609   0.001728            73.712737           369
  EURUSD       XGBoost   0.000928             0.001682   0.001797            74.254743           369
  GBPUSD Random Forest   0.000610             0.001359   0.001528            70.460705           369
  GBPUSD       XGBoost   0.000552             0.001353   0.001575            67.479675           369
  USDJPY Random Forest   0.001662             0.002918   0.003200            73.983740           369
  USDJPY       XGBoost   0.001742             0.003008   0.003282            73.170732           369
Forecast bias analysis:
Currency         Model  Mean_Bias  Mean_Absolute_Error  Error_Std  Overprediction_Rate  Observations
  EURUSD Random Forest   0.000831             0.001609   0.001728            73.712737           369
  EURUSD       XGBoost   0.000928          

In [ ]:
# Average out-of-sample performance across folds
test_summary = (
    performance
    .groupby(["Currency", "Model"])
    .agg(
        Test_RMSE=("RMSE", "mean"),
        Test_MAE=("MAE", "mean"),
        Test_R2=("R2", "mean")
    )
    .reset_index()
)

# Average training performance
train_summary = (
    training_performance
    .groupby(["Currency", "Model"])
    .agg(
        Train_RMSE=("Train_RMSE", "mean"),
        Train_R2=("Train_R2", "mean")
    )
    .reset_index()
)

# Average generalisation gap
gap_summary = (
    generalisation
    .groupby(["Currency", "Model"])
    .agg(
        RMSE_Gap=("RMSE_Gap", "mean"),
        RMSE_Ratio=("RMSE_Ratio", "mean")
    )
    .reset_index()
)

# Combine all performance measures
scorecard = (
    test_summary
    .merge(train_summary, on=["Currency", "Model"])
    .merge(gap_summary, on=["Currency", "Model"])
    .merge(
        bias_analysis[
            ["Currency", "Model", "Mean_Bias",
             "Overprediction_Rate"]
        ],
        on=["Currency", "Model"]
    )
)

# Rank models wiithin each currency pair by test RMSE
scorecard["RMSE_Rank"] = (
    scorecard.groupby("Currency")["Test_RMSE"]
    .rank(method="min")
    .astype(int)
)

# Display scorecard
print("Final model selection scorecard:")

print(scorecard.round(6).to_string(index=False))

# Save scorecard
scorecard.to_csv( "../outputs/evaluation/final_model_selection_scorecard.csv", index=False)
print("\nScorecard saved successfully")

Final model selection scorecard:


AttributeError: 'int' object has no attribute 'to_string'